In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from google.cloud import storage

In [2]:
credentials_location = '/Users/jishnuvsarmah/.google/credentials/google_credentials.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [3]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

26/04/16 00:33:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [6]:
bucket_name = "terraform-demo-472408-terra-bucket"
prefix = "pq/green/"

client = storage.Client.from_service_account_json(credentials_location)
bucket = client.get_bucket(bucket_name)

# Get all unique year/month folder paths
blobs = client.list_blobs(bucket_name, prefix=prefix)
paths = set()
for blob in blobs:
    # blob.name looks like pq/green/2020/01/file.parquet
    parts = blob.name.split("/")
    if len(parts) >= 4:
        folder = f"gs://{bucket_name}/{parts[0]}/{parts[1]}/{parts[2]}/{parts[3]}/"
        paths.add(folder)

paths = list(paths)
print(paths)

['gs://terraform-demo-472408-terra-bucket/pq/green/2021/01/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2021/08/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2021/03/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2020/08/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2020/04/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2021/06/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2021/02/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2020/07/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2021/04/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2021/05/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2020/02/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2020/03/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2021/07/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2020/05/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2020/06/', 'gs://terraform-demo-472408-terra-bucket/pq/green/2020/11/', 'gs://terraform-demo-47

In [7]:
df_green = spark.read.parquet(*paths)

In [8]:
df_green.show()

[Stage 1:>                                                          (0 + 1) / 1]

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2020-01-12 18:15:04|  2020-01-12 18:19:52|                 N|         1|          41|          41|              1|         0.78|        5.5|  0.0|    0.